# HW02-2：房地产上市公司财务特征分析

**状态：待 CSMAR 授权数据。当前文件已准备分析方法、审计和绘图程序，尚无真实样本、统计结果和实证结论，不是可提交的完成稿。**

- 姓名：毛博平
- 课程编号：待按教师要求填写（公开版不展示完整学号）
- 作业要求：https://lianxhcn.github.io/FinEco/exercises/hw-02.html
- 仓库地址：https://github.com/ElwinCamnlmen/FinEco-hw
- HW02-1 入口：https://github.com/ElwinCamnlmen/FinEco-hw/tree/main/hw02-1
- HW02-2 入口：https://github.com/ElwinCamnlmen/FinEco-hw/tree/main/hw02-2
- 数据来源：CSMAR，学校授权；下载日期、数据表、字段定义及单位待实际下载后记录。
- 样本期间：2005—2015 年；2004 年末财务数据用于平均分母。
- AI 使用声明：使用 OpenAI Codex 准备代码和报告结构；真实数据和本人核验未完成。

本题以各年年末仍上市且归属房地产行业的 A 股公司为研究对象，建立企业—年度非平衡面板，统计数量与当年实际控制人产权结构，比较六项财务指标的均值、中位数及国有、民营差异。先按历史身份建立公司名单，再左连接财务表，避免财务缺失使公司数量被低估。

## 1. 数据来源、字段和范围

### 分析说明
原始下载保留不改；根据同包字段说明和分类字典确认数据含义，再形成五张标准表。标准化层保留源文件指纹、原字段和单位转换，不把源记录的重复、缺失或历史口径问题隐藏在重命名里。标准表仅为程序输入约定，**不能用人工编造的表替代 CSMAR 数据**。

- `company_year`：当年年末证券身份、上市状态、历史行业及分类版本。
- `ownership_year`：当年实际控制人性质及已确认的国有、民营、其他、不明映射。
- `balance`：2004—2015 年末年度合并资产负债表。
- `income`：2005—2015 年度合并净利润。
- `loans`：2005—2015 年末短期与长期银行借款，须核对字段定义。

报表日期、A/B 报表类型、重复版本和原始单位须在标准化前核查。利润和权益统一为包含少数股东部分的合并口径；不以“非国有”识别民营。历史公司范围须包含研究期内存在、后来退市或转行业的公司。字段对应表和下载条件随数据保留在本地。

In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager,ticker
from IPython.display import display

candidates=[Path.cwd(),Path.cwd()/'hw02-2',Path.cwd()/'FinEco-hw/hw02-2',Path.cwd().parent/'FinEco-hw/hw02-2']
BASE=next((p for p in candidates if (p/'code/panel_analysis.py').is_file()),candidates[0])
assert (BASE/'code/panel_analysis.py').is_file(), '请从本题或仓库根目录运行'
sys.path.insert(0,str(BASE/'code'))
from panel_analysis import load_inputs,require_unique,SCHEMAS,KEY,YEARS,OWNERS,METRICS
DATA=BASE/'data/normalized'; RESULT=BASE/'results'
required=[DATA/f'{k}.csv' for k in SCHEMAS]+[DATA/'source_manifest.json',DATA/'field_mapping.csv']
missing=[p.name for p in required if not p.is_file()]
DATA_READY=not missing
if not DATA_READY:
    print('未执行实证分析。尚缺：'+', '.join(missing))
    print('当前为待数据稿；不使用模拟值、当前行业或当前产权回填历史。')
else:
    tables,source_manifest,field_mapping=load_inputs(DATA)
    print('读取获授权的本地快照；字段与口径验证通过。')
    display(field_mapping)
    RESULT.mkdir(exist_ok=True)
    font_names={f.name for f in font_manager.fontManager.ttflist}
    font=next((n for n in ['Arial Unicode MS','PingFang SC','Microsoft YaHei','SimHei','Noto Sans CJK SC'] if n in font_names),None)
    assert font, '请安装中文字体'
    plt.rcParams.update({'font.family':font,'axes.unicode_minus':False,'axes.spines.top':False,
                         'axes.spines.right':False,'figure.dpi':110,'axes.grid':True,'grid.alpha':0.2})

未执行实证分析。尚缺：company_year.csv, ownership_year.csv, balance.csv, income.csv, loans.csv, source_manifest.json, field_mapping.csv
当前为待数据稿；不使用模拟值、当前行业或当前产权回填历史。


### 结果解读
尚未收到原始 CSMAR 文件，不能报告实际公司覆盖数、字段可用性或缺失比例。拿到数据后，应先完成下载条件和原字段对应表，再填写本节。当前行业、当前产权和“股票仍然存在”的信息均不能替代历史年度身份。

## 2. 公司年度样本与合并审计

### 分析说明
一行代表一家公司一年，证券代码与年度组合唯一。先筛当年年末的房地产 A 股公司，再以一对一左连接合并产权、资产负债表、利润表和银行借款。财务未匹配保留在公司名单中，并逐年报告未匹配数量。

- 重复主键直接停止，不使用 `drop_duplicates` 或取第一行掩盖问题。
- 标准化需先处理报表版本，保留真实决策及受影响记录数；跨表金额统一为元。
- 上期资产、权益通过“上年年份加一后按公司—年连接”，防止跨缺年移位。
- 不强制每家公司在全部 11 年出现，允许样本进入、退出和产权变更。

In [2]:
def build_panel(tables):
    """先建历史样本，再左连接；财务缺失不改变公司数量。"""
    for name,df in tables.items(): require_unique(df,name)
    stages=[]
    def record(label,df): stages.append({'阶段':label,'公司数':df.stock_code.nunique(),'企业年度数':len(df)})
    for name,df in tables.items(): record('标准化输入：'+name,df)
    universe=tables['company_year']
    sample=universe.loc[universe.year.between(2005,2015)&universe.is_a_share.eq(1)&universe.listed_eoy.eq(1)&universe.is_real_estate.eq(1)].copy()
    record('当年年末房地产A股样本',sample)
    if sample.empty: raise ValueError('样本为空，请核对历史身份筛选')
    if not set(YEARS).issubset(set(sample.year)): raise ValueError('2005—2015 年并非每年都有样本，先核对下载范围')
    unmatched=[]
    for name in ['ownership_year','balance','income','loans']:
        sample=sample.merge(tables[name],on=KEY,how='left',validate='one_to_one',indicator=f'_merge_{name}')
        unmatched.append(sample.groupby('year')[f'_merge_{name}'].apply(lambda x:int(x.eq('left_only').sum())).rename(name))
        record('左连接：'+name,sample)
    sample['ownership']=sample.ownership.fillna('不明')
    # 让上年记录的年份加一后合并：缺 2006 不会把 2005 当 2007 的上期。
    prior=tables['balance'][KEY+['total_assets','total_equity']].copy()
    prior['year']=prior['year']+1
    prior=prior.rename(columns={'total_assets':'assets_lag','total_equity':'equity_lag'})
    sample=sample.merge(prior,on=KEY,how='left',validate='one_to_one',indicator='_merge_prior')
    record('连接相邻上年末分母',sample)
    return sample,pd.DataFrame(stages),pd.concat(unmatched,axis=1).reindex(YEARS,fill_value=0)

if DATA_READY:
    panel,sample_flow,unmatched=build_panel(tables)
    display(sample_flow); display(unmatched)
    print('上年末财务记录未匹配数：',int(panel['_merge_prior'].eq('left_only').sum()))
    sample_flow.to_csv(RESULT/'sample_flow.csv',index=False,encoding='utf-8-sig')
    unmatched.to_csv(RESULT/'unmatched_by_year.csv',encoding='utf-8-sig')

### 结果解读
真实数据接入后，本节须报告原始下载、筛选、逐表合并和变量构造后的公司数及企业—年度数，区分财务缺失与身份不确定。未匹配率必须使用对应年度或组别的样本数作分母，不能把全部样本作为每个组的分母。当前未填任何样本数量。

## 3. 六项指标与无效分母

### 分析说明
资产负债率为总负债/总资产；银行借款占比为（短期银行借款+长期银行借款）/总负债；短期负债占比为流动负债/总负债；ROA、ROE 分别用合并净利润除以相邻年末平均总资产、平均合并权益；现金持有为货币资金/总资产。

- 资产和负债分母必须为正。负净利润有经济含义，保留；资产负债率超过 1 不自动删除。
- ROE 主分析要求本年及上年权益均为正，避免负权益下比率符号反转；保留原值，用允许非零符号分母的口径作敏感性比较。
- 银行借款任一字段缺失时，银行借款合计仍缺失；零值只有原表明确记录零才能使用。
- 货币资金、流动负债和银行借款的负值影响相应指标，不删除整家公司年度。违反资产负债表勾稽关系、分子大于其上位总量的记录另行审计，未核清前不宣称数据全部合理。

In [3]:
def make_metrics(panel):
    """主分析不缩尾。无效分母只影响对应指标，不删除公司年度。"""
    p=panel.copy()
    ta=p.total_assets.where(p.total_assets.gt(0))
    tl=p.total_liabilities.where(p.total_liabilities.gt(0))
    p['avg_assets']=(p.total_assets+p.assets_lag)/2
    p['avg_equity']=(p.total_equity+p.equity_lag)/2
    avgta=p.avg_assets.where(p.total_assets.gt(0)&p.assets_lag.gt(0))
    avgeq=p.avg_equity.where(p.total_equity.gt(0)&p.equity_lag.gt(0))
    p['leverage']=p.total_liabilities.where(p.total_liabilities.ge(0))/ta
    # 加总保留任一借款字段的缺失，不把空白认作无银行借款。
    bank=p[['short_bank_loan','long_bank_loan']].sum(axis=1,min_count=2)
    bank=bank.where(p.short_bank_loan.ge(0)&p.long_bank_loan.ge(0))
    p['bankloan']=bank/tl
    p['short_debt']=p.current_liabilities.where(p.current_liabilities.ge(0))/tl
    p['ROA']=p.net_income/avgta
    p['ROE']=p.net_income/avgeq
    p['Cash_TA']=p.cash.where(p.cash.ge(0))/ta
    # ROE敏感性：允许符号分母，但排除零和缺失，不把正比率等同盈利。
    p['ROE_signed']=p.net_income/p.avg_equity.where(p.avg_equity.ne(0))
    flags={
      '总资产非正':p.total_assets.le(0),'总负债非正':p.total_liabilities.le(0),
      '任一相邻年权益非正':p.total_equity.le(0)|p.equity_lag.le(0),
      '上年资产缺失':p.assets_lag.isna(),'上年权益缺失':p.equity_lag.isna(),
      '任一银行借款字段缺失':p[['short_bank_loan','long_bank_loan']].isna().any(axis=1),
      '流动负债大于总负债':p.current_liabilities.gt(p.total_liabilities),
      '银行借款大于总负债':bank.gt(p.total_liabilities),
      '货币资金大于总资产':p.cash.gt(p.total_assets),
      '资产不等于负债加权益':(p.total_assets-p.total_liabilities-p.total_equity).abs().gt(p.total_assets.abs()*0.001+1),
    }
    audit=pd.DataFrame({k:v.groupby(p.year).sum().astype(int) for k,v in flags.items()}).reindex(YEARS)
    for k,v in flags.items(): p['flag_'+k]=v
    audit_missing=p.groupby('year')[list(METRICS)].apply(lambda d:d.isna().sum()).reindex(YEARS)
    return p,audit,audit_missing

if DATA_READY:
    panel,denominator_audit,missing_metrics=make_metrics(panel)
    display(denominator_audit); display(missing_metrics)
    denominator_audit.to_csv(RESULT/'denominator_audit.csv',encoding='utf-8-sig')
    missing_metrics.to_csv(RESULT/'missing_by_metric_year.csv',encoding='utf-8-sig')
    # 公司级记录仅保存在本地受限目录，不进入公开仓库。
    private_path=BASE/'data/private_derived'; private_path.mkdir(exist_ok=True)
    panel.to_csv(private_path/'panel_with_flags.csv',index=False,encoding='utf-8-sig')

### 结果解读
待真实数据运行后，逐项说明零分母、负权益、上期缺失和异常分子的影响数量，以及哪些指标因此少于公司总数。ROE 主分析与符号分母口径的比较必须使用相同财务来源，不把负权益下的正 ROE 解读成盈利。当前不预设异常规模，也不预先声称缩尾必要。

## 4. 公司数量与产权结构

### 分析说明
逐年按公司去重计数，国有、民营、其他、不明四组之和等于当年房地产公司总数。国有与民营比例分别除以总数；存在其他、不明时，两者之和可以小于 100%。公司数量使用历史身份名单，不以六项财务指标完整为前提。

In [4]:
def summarize(p):
    counts=p.groupby(['year','ownership']).stock_code.nunique().unstack(fill_value=0).reindex(index=YEARS,columns=OWNERS,fill_value=0)
    counts['总数']=counts[OWNERS].sum(axis=1)
    counts['国有占比']=counts['国有']/counts['总数']; counts['民营占比']=counts['民营']/counts['总数']
    counts['其他或不明占比']=(counts['其他']+counts['不明'])/counts['总数']
    long=p.melt(id_vars=KEY+['ownership'],value_vars=list(METRICS),var_name='metric',value_name='value')
    overall=long.groupby(['year','metric']).value.agg(['count','mean','median']).reset_index()
    groups=long.groupby(['year','ownership','metric']).value.agg(['count','mean','median']).reset_index()
    # 保留缺席产权组的零有效样本量，而不是让该行静默消失。
    fullidx=pd.MultiIndex.from_product([YEARS,OWNERS,list(METRICS)],names=['year','ownership','metric'])
    groups=groups.set_index(['year','ownership','metric']).reindex(fullidx).reset_index()
    groups['count']=groups['count'].fillna(0).astype(int)
    return counts,overall,groups,long

if DATA_READY:
    counts,overall,groups,long=summarize(panel)
    assert counts['总数'].eq(panel.groupby('year').stock_code.nunique()).all()
    display(counts)
    fig,axes=plt.subplots(1,3,figsize=(16,4.5))
    counts['总数'].plot(ax=axes[0],marker='o',color='#137C8B',label='房地产公司总数')
    counts[OWNERS].plot(ax=axes[1],marker='o')
    counts[['国有占比','民营占比']].plot(ax=axes[2],marker='o')
    for ax,title in zip(axes,['房地产上市公司总数','按当年实际控制人性质分组','国有、民营占全部样本的比例']):
        ax.set_title(title); ax.set_xlabel('年度'); ax.legend(fontsize=8)
    axes[0].set_ylabel('公司数 / 家'); axes[1].set_ylabel('公司数 / 家')
    axes[2].set_ylabel('占当年总数比例'); axes[2].yaxis.set_major_formatter(ticker.PercentFormatter(1))
    fig.tight_layout(); fig.savefig(RESULT/'01_counts_ownership.png',dpi=150,bbox_inches='tight'); plt.show()
    counts.to_csv(RESULT/'company_counts.csv',encoding='utf-8-sig')

### 结果解读
待运行后引用 2005 年、2015 年以及变化突出的年份，解释公司总数与产权占比的变化。不能把公司数增加全部归因于新上市，也可能是行业进入或分类映射变化；“退出房地产样本”同样不直接等于退市。需要结合年度进入退出和产权变化诊断。

## 5. 六项财务指标的均值、中位数及产权差异

### 分析说明
每个年度对公司层面的比率做算术平均及中位数汇总，不采用行业总量之比。三组六面板图分别显示全样本均值与中位数、国有与民营均值、国有与民营中位数。每个指标单独计算有效样本数，不能用一个统一的 N 代替六项指标的可用样本。

In [5]:
if DATA_READY:
    valid_n=overall.pivot(index='year',columns='metric',values='count').reindex(YEARS)
    valid_group_n=groups.pivot(index=['year','ownership'],columns='metric',values='count')
    print('全样本各指标有效公司数'); display(valid_n)
    print('各产权组各指标有效公司数'); display(valid_group_n)
    for mode,title,filename in [('overall','全部房地产企业：均值与中位数','02_overall.png'),
            ('mean','国有与民营企业：年度均值','03_group_mean.png'),
            ('median','国有与民营企业：年度中位数','04_group_median.png')]:
        fig,axes=plt.subplots(3,2,figsize=(13,11),sharex=True)
        for ax,(key,label) in zip(axes.flat,METRICS.items()):
            if mode=='overall':
                data=overall[overall.metric.eq(key)].set_index('year')
                ax.plot(data.index,data['mean'],marker='o',label='均值',color='#137C8B')
                ax.plot(data.index,data['median'],marker='s',label='中位数',color='#D57832')
            else:
                for owner,color in [('国有','#137C8B'),('民营','#D57832')]:
                    data=groups[groups.metric.eq(key)&groups.ownership.eq(owner)].set_index('year')
                    ax.plot(data.index,data[mode],marker='o',label=owner,color=color)
            ax.set_title(label); ax.set_ylabel('比率'); ax.yaxis.set_major_formatter(ticker.PercentFormatter(1)); ax.legend()
        for ax in axes[-1]: ax.set_xlabel('年度')
        fig.suptitle(title,fontsize=16); fig.tight_layout()
        fig.savefig(RESULT/filename,dpi=150,bbox_inches='tight'); plt.show()
    valid_n.to_csv(RESULT/'valid_n_overall.csv',encoding='utf-8-sig')
    valid_group_n.to_csv(RESULT/'valid_n_ownership.csv',encoding='utf-8-sig')
    overall.to_csv(RESULT/'financial_overall.csv',index=False,encoding='utf-8-sig')
    groups.to_csv(RESULT/'financial_ownership.csv',index=False,encoding='utf-8-sig')

### 结果解读
待实际计算后，每项指标至少说明起点、终点和重要拐点，比较均值和中位数是否同步变化，引用国有与民营的实际差值及各自有效样本量。均值受极端值影响较大，中位数可以补充“典型企业”情况。产权分组只是描述性比较，没有控制规模、地区、经营阶段等混杂因素，不能解释为产权的因果作用。

## 6. 离群值、样本变化与稳健性核验

### 分析说明
主分析保留满足定义的极端比率，标记而不自动删除。对年度 1%/99% 缩尾、ROE 的符号分母口径做对照，报告受影响观测数及均值、中位数变化；有效观测不足 20 时不缩尾，避免对很小样本作机械修整。诊断逐年样本进入、退出和连续企业产权变更，寻找图形突变的来源。

In [6]:
def sensitivity(p):
    # 每年每指标 1%/99% 缩尾只作为对照；不足 20 个有效观测不缩尾。
    rows=[]
    for year,g in p.groupby('year'):
        for key in METRICS:
            v=g[key].dropna(); n=len(v)
            low,high=v.quantile([.01,.99]) if n>=20 else (-np.inf,np.inf)
            clipped=v.clip(low,high)
            rows.append({'year':year,'metric':key,'n':n,'raw_mean':v.mean(),'winsor_mean':clipped.mean(),
                         'raw_median':v.median(),'winsor_median':clipped.median(),
                         'affected':int(v.ne(clipped).sum())})
    roe=p.groupby('year')[['ROE','ROE_signed']].agg(['count','mean','median'])
    # 进入/退出/产权变更诊断：退出是从房地产样本退出，不直接等同退市。
    movement=[]
    for year in range(2006,2016):
        before=p[p.year.eq(year-1)].set_index('stock_code'); now=p[p.year.eq(year)].set_index('stock_code')
        common=before.index.intersection(now.index)
        changes=(before.loc[common,'ownership']!=now.loc[common,'ownership']).sum()
        movement.append({'year':year,'进入本年房地产样本':len(now.index.difference(before.index)),
                         '退出本年房地产样本':len(before.index.difference(now.index)),
                         '连续两年在样本的公司':len(common),'产权分类变化':int(changes)})
    return pd.DataFrame(rows),roe,pd.DataFrame(movement)

if DATA_READY:
    winsor_comparison,roe_comparison,movement=sensitivity(panel)
    display(winsor_comparison); display(roe_comparison); display(movement)
    extrema=long.groupby(['year','metric']).value.agg(['count','min','max',lambda v:v.quantile(.01),lambda v:v.quantile(.99)])
    extrema.columns=['n','min','max','p01','p99']
    display(extrema)
    winsor_comparison.to_csv(RESULT/'winsor_comparison.csv',index=False,encoding='utf-8-sig')
    roe_comparison.to_csv(RESULT/'roe_denominator_comparison.csv',encoding='utf-8-sig')
    movement.to_csv(RESULT/'sample_movement.csv',index=False,encoding='utf-8-sig')
    extrema.to_csv(RESULT/'extreme_diagnostics.csv',encoding='utf-8-sig')

### 结果解读与总体结论
目前尚无 CSMAR 真实数据，不能判断企业数量、产权结构或六个财务指标的变化方向，也不能写“国有显著高于民营”等结论。数据到位后，本节必须结合实际审计和图表写成具体结论，并说明主要结果是否依赖分母、缩尾或样本变化；若存在无法解释的源数据问题，应保留疑点和影响范围。

当前已明确的研究判断是：使用历史年度身份与非平衡面板；财务缺失不删公司计数；只匹配相邻年份；银行借款缺失不填零；主 ROE 采用相邻两年正权益样本并做口径比较；不将描述性差异解释为因果作用。这些方法不能替代实际结果。

## 7. 数据权限、复现与 AI 使用

原始 CSMAR 数据、标准化公司级记录及派生公司级面板仅留在获授权的本地目录。公开仓库提供 Notebook、程序、依赖和下载条件；在学校及数据库许可范围内向教师受控提供数据。未经许可的第三方不能仅凭公开仓库访问 CSMAR 数据。`data/README.md` 说明文件、用途和相对路径；获取日期、文件大小及授权访问方式在收到文件后补齐。

AI 工具为 OpenAI Codex，参与数据流程设计、程序及文档草稿。关键提示词：“按各年年末上市、行业和实际控制人性质建立 2005—2015 房地产 A 股非平衡面板，额外读取 2004 年末分母，严格一对一合并，按题目计算六项指标；缺数据时明确停止，不虚构结果。”

当前程序尚未在用户真实 CSMAR 数据上验证。程序边界用另存于工作目录的教学小表检查，不将其输出作为作业结果。本人核验尚未完成，待真实数据运行后由毛博平核对样本规则、至少一家企业的六项指标及主要解释，再如实补充核验记录。来源：[教师作业要求](https://lianxhcn.github.io/FinEco/exercises/hw-02.html)、[CSMAR 平台](https://data.csmar.com/)。